# Sarah Alhazeem — Data Literacy Portfolio

This notebook generates the portfolio webpage `docs/index.html` deployed via GitHub Pages.

**Skills demonstrated (70+ marking criteria):**
- Python programming (Weeks 1-4)
- Data visualisation with matplotlib and pandas (Weeks 1-4, 6)
- Real financial data retrieval using yfinance (self-exploration)
- Data wrangling: resampling, pct_change, rolling statistics (Weeks 6-9)
- Risk analysis: volatility, drawdown, returns (financial self-exploration)
- Version control with GitHub (Week 10)
- Interactive web publishing via GitHub Pages (self-exploration)

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import base64
import io
import pathlib
print('Libraries imported.')

## Chart Style and Helper Function

In [ ]:
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.facecolor': '#f8fbff',
    'axes.facecolor': '#f8fbff',
})

def fig_to_base64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=130, bbox_inches='tight')
    buf.seek(0)
    encoded = base64.b64encode(buf.read()).decode('utf-8')
    plt.close(fig)
    return encoded

print('Style configured.')

## Step 1 — Download Real Stock Data from Yahoo Finance

In [ ]:
print('Downloading real stock data from Yahoo Finance...')
aapl = yf.download('AAPL', start='2020-01-01', end='2024-12-31', auto_adjust=True, progress=False)
tsla = yf.download('TSLA', start='2020-01-01', end='2024-12-31', auto_adjust=True, progress=False)

aapl_close = aapl['Close'].squeeze()
tsla_close = tsla['Close'].squeeze()

print(f'Apple: {len(aapl_close)} data points, latest: ${aapl_close.iloc[-1]:.2f}')
print(f'Tesla: {len(tsla_close)} data points, latest: ${tsla_close.iloc[-1]:.2f}')
print(aapl_close.tail(3))

## Step 2 — Compute Key Financial Statistics

In [ ]:
aapl_annual = float(aapl_close.iloc[-1] / aapl_close.iloc[0] - 1) * 100
tsla_annual = float(tsla_close.iloc[-1] / tsla_close.iloc[0] - 1) * 100
aapl_vol_val = float(aapl_close.pct_change().dropna().std() * np.sqrt(252) * 100)
tsla_vol_val = float(tsla_close.pct_change().dropna().std() * np.sqrt(252) * 100)
aapl_max_dd = float(((aapl_close / aapl_close.cummax()) - 1).min() * 100)
tsla_max_dd = float(((tsla_close / tsla_close.cummax()) - 1).min() * 100)

stats = pd.DataFrame({
    'Total Return (%)': [round(aapl_annual,1), round(tsla_annual,1)],
    'Annualised Volatility (%)': [round(aapl_vol_val,1), round(tsla_vol_val,1)],
    'Max Drawdown (%)': [round(aapl_max_dd,1), round(tsla_max_dd,1)]
}, index=['Apple (AAPL)', 'Tesla (TSLA)'])

print(stats)

## Step 3 — Chart 1: Stock Price Over Time (Apple vs Tesla)

In [ ]:
fig1, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(aapl_close.index, aapl_close.values, color='#2e86c1', linewidth=1.8, label='Apple (AAPL)')
ax1.plot(tsla_close.index, tsla_close.values, color='#e74c3c', linewidth=1.8, label='Tesla (TSLA)', linestyle='--')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.xticks(rotation=30, ha='right', fontsize=8)
ax1.set_title('Apple vs Tesla - Real Stock Price (2020-2024)', fontsize=12, fontweight='bold', pad=12)
ax1.set_ylabel('Closing Price (USD)', fontsize=9)
ax1.legend(fontsize=9)
fig1.tight_layout()
CHART1 = fig_to_base64(fig1)
print('Chart 1 generated.')

## Step 4 — Chart 2: Monthly Returns Apple vs Tesla (2023)

In [ ]:
aapl_monthly = aapl_close.resample('ME').last().pct_change().dropna() * 100
tsla_monthly = tsla_close.resample('ME').last().pct_change().dropna() * 100
df_monthly = pd.DataFrame({'AAPL': aapl_monthly, 'TSLA': tsla_monthly}).dropna()
df_2023 = df_monthly[df_monthly.index.year == 2023]

x = np.arange(len(df_2023))
months_labels = [d.strftime('%b') for d in df_2023.index]
fig2, ax2 = plt.subplots(figsize=(9, 4))
ax2.bar(x - 0.175, df_2023['AAPL'], 0.35, label='Apple', color='#2e86c1', alpha=0.85)
ax2.bar(x + 0.175, df_2023['TSLA'], 0.35, label='Tesla', color='#e74c3c', alpha=0.85)
ax2.axhline(0, color='#555', linewidth=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels(months_labels, fontsize=8)
ax2.set_title('Monthly Returns - Apple vs Tesla (2023)', fontsize=12, fontweight='bold', pad=12)
ax2.set_ylabel('Monthly Return (%)', fontsize=9)
ax2.legend(fontsize=9)
fig2.tight_layout()
CHART2 = fig_to_base64(fig2)
print('Chart 2 generated.')

## Step 5 — Chart 3: S&P 500 Sector Returns 2023

In [ ]:
sectors = ['Technology', 'Healthcare', 'Financials', 'Energy', 'Consumer Disc.']
returns_2023 = [56.4, 2.1, 10.1, -4.8, 41.3]
colors = ['#2e86c1' if r > 0 else '#e74c3c' for r in returns_2023]

fig3, ax3 = plt.subplots(figsize=(8, 4))
bars = ax3.bar(sectors, returns_2023, color=colors, width=0.5, edgecolor='white', linewidth=1.2)
ax3.axhline(0, color='#555', linewidth=0.8)
for bar, val in zip(bars, returns_2023):
    ax3.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + (0.8 if val >= 0 else -2.5),
             f'{val:+.1f}%', ha='center', va='bottom', fontsize=8.5, fontweight='bold')
ax3.set_title('S&P 500 Sector Returns - 2023 (%)', fontsize=12, fontweight='bold', pad=12)
ax3.set_ylabel('Annual Return (%)', fontsize=9)
ax3.set_ylim(-12, 68)
fig3.tight_layout()
CHART3 = fig_to_base64(fig3)
print('Chart 3 generated.')

## Step 6 — Chart 4: 30-Day Rolling Volatility

In [ ]:
aapl_roll_vol = aapl_close.pct_change().dropna().rolling(30).std() * np.sqrt(252) * 100
tsla_roll_vol = tsla_close.pct_change().dropna().rolling(30).std() * np.sqrt(252) * 100

fig4, ax4 = plt.subplots(figsize=(9, 4))
ax4.plot(aapl_roll_vol.index, aapl_roll_vol.values, color='#2e86c1', linewidth=1.8, label='Apple (AAPL)')
ax4.plot(tsla_roll_vol.index, tsla_roll_vol.values, color='#e74c3c', linewidth=1.8, label='Tesla (TSLA)', linestyle='--')
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax4.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.xticks(rotation=30, ha='right', fontsize=8)
ax4.set_title('30-Day Rolling Volatility - Apple vs Tesla (2020-2024)', fontsize=12, fontweight='bold', pad=12)
ax4.set_ylabel('Annualised Volatility (%)', fontsize=9)
ax4.legend(fontsize=9)
fig4.tight_layout()
CHART4 = fig_to_base64(fig4)
print('Chart 4 generated.')

## Step 7 — Define Profile and Skills Content

In [ ]:
profile = {
    'name': 'Sarah Alhazeem',
    'title': 'Accounting & Finance Student | Data Literacy Enthusiast',
    'github': 'salhazeem',
    'repo': 'repo2-AF1204-salhazeem',
    'about': (
        'Hi, I am Sarah, an Accounting and Finance student with a growing passion for '
        'data literacy and financial analytics. Through my studies and hands-on module work, '
        'I have developed skills in Python programming, data visualisation, and interpreting '
        'real financial market data. I enjoy bridging the gap between traditional finance '
        'and modern data tools, turning raw stock data into meaningful insights about '
        'market behaviour and risk.'
    ),
}

skills = [
    {'icon': 'P', 'name': 'Python Programming',           'level': 75, 'desc': 'Data manipulation, scripting, and analysis using Python fundamentals.'},
    {'icon': 'C', 'name': 'Data Visualisation',            'level': 80, 'desc': 'Creating charts using matplotlib and pandas to communicate financial insights.'},
    {'icon': 'F', 'name': 'Financial Data Interpretation', 'level': 85, 'desc': 'Reading and analysing stock data, returns, volatility, and market indicators.'},
    {'icon': 'G', 'name': 'Version Control (GitHub)',      'level': 70, 'desc': 'Managing code with Git, commits, branches, and collaborative workflows.'},
    {'icon': 'J', 'name': 'Jupyter Notebook',              'level': 75, 'desc': 'Building reproducible notebooks for data analysis and portfolio generation.'},
    {'icon': 'D', 'name': 'Pandas & Data Wrangling',       'level': 70, 'desc': 'Cleaning, resampling, and exploring financial datasets using pandas.'},
]

print(f'Profile: {profile["name"]}')
print(f'Skills defined: {len(skills)}')

## Step 8 — Generate HTML and Write to docs/index.html

In [ ]:
CSS = '''
    :root{--primary:#1a3c5e;--accent:#2e86c1;--light:#eaf4fb;--text:#2c3e50;--muted:#7f8c8d;--card-bg:#ffffff;--border:#d5e8f5;}
    *{box-sizing:border-box;margin:0;padding:0;}
    body{font-family:Segoe UI,Tahoma,Geneva,Verdana,sans-serif;background:#f0f6fc;color:var(--text);line-height:1.7;}
    nav{background:var(--primary);padding:0 2rem;display:flex;align-items:center;justify-content:space-between;position:sticky;top:0;z-index:100;box-shadow:0 2px 8px rgba(0,0,0,0.2);}
    nav .logo{color:#fff;font-size:1.2rem;font-weight:700;padding:1rem 0;letter-spacing:1px;}
    nav ul{list-style:none;display:flex;gap:1.5rem;}
    nav ul a{color:#cce4f7;text-decoration:none;font-size:0.95rem;transition:color 0.2s;}
    nav ul a:hover{color:#fff;}
    .hero{background:linear-gradient(135deg,var(--primary) 0%,var(--accent) 100%);color:#fff;padding:5rem 2rem 4rem;text-align:center;}
    .hero-avatar{width:110px;height:110px;border-radius:50%;background:rgba(255,255,255,0.15);border:3px solid rgba(255,255,255,0.5);display:flex;align-items:center;justify-content:center;font-size:3rem;margin:0 auto 1.5rem;}
    .hero h1{font-size:2.4rem;font-weight:700;margin-bottom:0.4rem;}
    .hero .subtitle{font-size:1.1rem;opacity:0.85;margin-bottom:1.5rem;}
    .hero .tags{display:flex;flex-wrap:wrap;gap:0.6rem;justify-content:center;}
    .hero .tags span{background:rgba(255,255,255,0.18);border:1px solid rgba(255,255,255,0.35);padding:0.3rem 0.9rem;border-radius:20px;font-size:0.85rem;}
    section{max-width:980px;margin:3rem auto;padding:0 1.5rem;}
    .section-title{font-size:1.5rem;font-weight:700;color:var(--primary);margin-bottom:1.5rem;padding-bottom:0.5rem;border-bottom:3px solid var(--accent);display:inline-block;}
    .about-card{background:var(--card-bg);border-radius:12px;padding:2rem;box-shadow:0 2px 12px rgba(0,0,0,0.07);border-left:5px solid var(--accent);}
    .skills-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(200px,1fr));gap:1.2rem;}
    .skill-card{background:var(--card-bg);border-radius:12px;padding:1.4rem 1.2rem;box-shadow:0 2px 10px rgba(0,0,0,0.07);text-align:center;transition:transform 0.2s,box-shadow 0.2s;}
    .skill-card:hover{transform:translateY(-4px);box-shadow:0 6px 18px rgba(0,0,0,0.12);}
    .skill-icon{font-size:2rem;margin-bottom:0.6rem;}
    .skill-card h3{font-size:0.95rem;font-weight:600;color:var(--primary);margin-bottom:0.3rem;}
    .skill-card p{font-size:0.82rem;color:var(--muted);}
    .skill-bars{margin-top:2rem;display:flex;flex-direction:column;gap:1rem;}
    .bar-item label{display:flex;justify-content:space-between;font-size:0.9rem;font-weight:600;color:var(--primary);margin-bottom:0.3rem;}
    .bar-track{background:var(--border);border-radius:20px;height:10px;overflow:hidden;}
    .bar-fill{height:100%;border-radius:20px;background:linear-gradient(90deg,var(--accent),#5dade2);}
    .project-card{background:var(--card-bg);border-radius:12px;padding:1.8rem;box-shadow:0 2px 10px rgba(0,0,0,0.07);border-top:4px solid var(--accent);margin-bottom:1.8rem;}
    .project-card h3{font-size:1.1rem;font-weight:700;color:var(--primary);margin-bottom:0.6rem;}
    .proj-desc{font-size:0.92rem;color:var(--text);margin-bottom:0.5rem;}
    .proj-tech{font-size:0.82rem;color:var(--accent);font-style:italic;margin-bottom:1rem;padding:0.4rem 0.8rem;background:var(--light);border-radius:6px;display:inline-block;}
    .chart-img{width:100%;border-radius:8px;margin-top:0.8rem;box-shadow:0 2px 8px rgba(0,0,0,0.08);}
    .chart-caption{font-size:0.82rem;color:var(--muted);margin-top:0.5rem;text-align:center;font-style:italic;}
    .proj-tags{display:flex;flex-wrap:wrap;gap:0.4rem;margin-top:1rem;}
    .proj-tags span{background:var(--light);color:var(--accent);font-size:0.75rem;padding:0.2rem 0.6rem;border-radius:12px;font-weight:600;border:1px solid var(--border);}
    .stats-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:1rem;margin:1rem 0;}
    .stat-box{background:var(--light);border-radius:8px;padding:0.8rem;text-align:center;border:1px solid var(--border);}
    .stat-box .stat-val{font-size:1.3rem;font-weight:700;color:var(--primary);}
    .stat-box .stat-lbl{font-size:0.75rem;color:var(--muted);}
    .interactive-box{background:var(--light);border-radius:10px;padding:1.2rem;margin-bottom:1rem;border:1px solid var(--border);}
    .interactive-box label{font-weight:600;color:var(--primary);font-size:0.9rem;margin-right:0.6rem;}
    .interactive-box select{padding:0.4rem 0.8rem;border-radius:6px;border:1.5px solid var(--accent);color:var(--primary);font-size:0.9rem;background:#fff;cursor:pointer;}
    .chart-panel{display:none;} .chart-panel.active{display:block;}
    .edu-card{background:var(--card-bg);border-radius:12px;padding:1.6rem;box-shadow:0 2px 10px rgba(0,0,0,0.07);display:flex;gap:1.2rem;align-items:flex-start;}
    .edu-icon{font-size:2.2rem;}
    .edu-card h3{font-size:1rem;font-weight:700;color:var(--primary);}
    .edu-card p{font-size:0.88rem;color:var(--muted);}
    .contact-card{background:linear-gradient(135deg,var(--primary),var(--accent));border-radius:12px;padding:2.5rem;text-align:center;color:#fff;}
    .contact-card h2{font-size:1.4rem;margin-bottom:0.5rem;}
    .contact-card p{opacity:0.85;margin-bottom:1.5rem;font-size:0.95rem;}
    .contact-links{display:flex;gap:1rem;justify-content:center;flex-wrap:wrap;}
    .contact-links a{background:rgba(255,255,255,0.15);border:1px solid rgba(255,255,255,0.35);color:#fff;padding:0.6rem 1.4rem;border-radius:25px;text-decoration:none;font-size:0.9rem;font-weight:600;transition:background 0.2s;}
    .contact-links a:hover{background:rgba(255,255,255,0.3);}
    footer{text-align:center;padding:2rem;color:var(--muted);font-size:0.85rem;border-top:1px solid var(--border);margin-top:3rem;}
'''

skill_icons = {'P':'&#128013;','C':'&#128202;','F':'&#128185;','G':'&#128295;','J':'&#128211;','D':'&#129518;'}

skill_cards = ''.join(
    f'<div class="skill-card"><div class="skill-icon">{skill_icons[s["icon"]]}</div>'
    f'<h3>{s["name"]}</h3><p>{s["desc"]}</p></div>' for s in skills)

skill_bars = ''.join(
    f'<div class="bar-item"><label><span>{s["name"]}</span><span>{s["level"]}%</span></label>'
    f'<div class="bar-track"><div class="bar-fill" style="width:{s["level"]}%"></div></div></div>' for s in skills)

print('HTML components ready.')

In [ ]:
html = f'''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8"/>
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <title>Sarah Alhazeem | Data Portfolio</title>
  <style>{CSS}</style>
</head>
<body>
<nav>
  <div class="logo">SA</div>
  <ul>
    <li><a href="#about">About</a></li>
    <li><a href="#skills">Skills</a></li>
    <li><a href="#projects">Projects</a></li>
    <li><a href="#education">Education</a></li>
    <li><a href="#contact">Contact</a></li>
  </ul>
</nav>
<div class="hero">
  <div class="hero-avatar">&#128105;&#8205;&#128188;</div>
  <h1>{profile["name"]}</h1>
  <p class="subtitle">{profile["title"]}</p>
  <div class="tags">
    <span>&#128202; Data Visualisation</span>
    <span>&#128013; Python</span>
    <span>&#128185; Financial Analysis</span>
    <span>&#128295; GitHub</span>
    <span>&#128200; S&amp;P 500</span>
  </div>
</div>
<section id="about">
  <div class="section-title">About Me</div>
  <div class="about-card"><p>{profile["about"]}</p></div>
</section>
<section id="skills">
  <div class="section-title">Skills</div>
  <div class="skills-grid">{skill_cards}</div>
  <div class="skill-bars">{skill_bars}</div>
</section>
<section id="projects">
  <div class="section-title">Projects</div>
  <div class="project-card">
    <h3>&#128200; Project 1 &mdash; Stock Price Analysis: Apple vs Tesla (2020&ndash;2024)</h3>
    <p class="proj-desc">This project analyses the real historical closing prices of Apple (AAPL) and Tesla (TSLA) downloaded directly from Yahoo Finance using the yfinance library. The line chart plots daily closing prices from January 2020 to December 2024, allowing a direct comparison of how both companies grew over five years. Apple shows a steady, consistent upward trend while Tesla displays much sharper rises and falls, reflecting its higher risk profile as a growth stock. This demonstrates how stock price data can be used to identify long-term trends and compare investment performance.</p>
    <span class="proj-tech">Created using Python &mdash; yfinance (real data), pandas, matplotlib</span>
    <div class="stats-grid">
      <div class="stat-box"><div class="stat-val">{aapl_annual:+.1f}%</div><div class="stat-lbl">Apple Total Return</div></div>
      <div class="stat-box"><div class="stat-val">{tsla_annual:+.1f}%</div><div class="stat-lbl">Tesla Total Return</div></div>
      <div class="stat-box"><div class="stat-val">2020&ndash;2024</div><div class="stat-lbl">Period Analysed</div></div>
    </div>
    <img class="chart-img" src="data:image/png;base64,{CHART1}" alt="Stock price chart"/>
    <p class="chart-caption">Figure 1 &mdash; Real daily closing prices for Apple and Tesla sourced from Yahoo Finance via yfinance</p>
    <div class="proj-tags"><span>Python</span><span>yfinance</span><span>Pandas</span><span>Matplotlib</span><span>Real Data</span></div>
  </div>
  <div class="project-card">
    <h3>&#128197; Project 2 &mdash; Monthly Return Comparison: Apple vs Tesla (2023)</h3>
    <p class="proj-desc">This project breaks down the month-by-month percentage returns of Apple and Tesla throughout 2023, calculated from real price data using pandas resampling and pct_change functions. The grouped bar chart reveals that Tesla's returns are far more volatile on a monthly basis with larger positive and negative swings compared to Apple's more stable performance. This analysis demonstrates how short-term volatility differs between growth stocks and established companies, a key concept in financial risk assessment.</p>
    <span class="proj-tech">Created using Python &mdash; yfinance (real data), pandas (resample, pct_change), matplotlib</span>
    <div class="stats-grid">
      <div class="stat-box"><div class="stat-val">{aapl_vol_val:.1f}%</div><div class="stat-lbl">Apple Annualised Volatility</div></div>
      <div class="stat-box"><div class="stat-val">{tsla_vol_val:.1f}%</div><div class="stat-lbl">Tesla Annualised Volatility</div></div>
      <div class="stat-box"><div class="stat-val">2023</div><div class="stat-lbl">Year Analysed</div></div>
    </div>
    <img class="chart-img" src="data:image/png;base64,{CHART2}" alt="Monthly returns chart"/>
    <p class="chart-caption">Figure 2 &mdash; Monthly percentage returns for Apple and Tesla in 2023 calculated from real closing price data</p>
    <div class="proj-tags"><span>Python</span><span>yfinance</span><span>Pandas</span><span>Matplotlib</span><span>Monthly Returns</span></div>
  </div>
  <div class="project-card">
    <h3>&#128202; Project 3 &mdash; S&amp;P 500 Sector Returns &amp; Rolling Volatility (Interactive)</h3>
    <p class="proj-desc">This project contains two interactive views. The first shows the annual returns of five major S&amp;P 500 sectors in 2023 based on publicly reported index data. Technology led with +56.4% while Energy was the only sector to decline at -4.8%. The second view shows the 30-day rolling annualised volatility of Apple and Tesla calculated from real daily returns using pandas rolling standard deviation, a standard risk metric used in finance. Use the dropdown below to switch between views.</p>
    <span class="proj-tech">Created using Python &mdash; pandas (rolling std, annualisation), matplotlib, real yfinance data</span>
    <div class="interactive-box">
      <label for="chartSelect">&#128270; Select view:</label>
      <select id="chartSelect" onchange="switchChart(this.value)">
        <option value="sector">S&amp;P 500 Sector Returns 2023</option>
        <option value="volatility">30-Day Rolling Volatility (Apple vs Tesla)</option>
      </select>
    </div>
    <div id="panel-sector" class="chart-panel active">
      <img class="chart-img" src="data:image/png;base64,{CHART3}" alt="Sector returns chart"/>
      <p class="chart-caption">Figure 3a &mdash; Annual returns by S&amp;P 500 sector for 2023. Technology and Consumer Discretionary led the market.</p>
    </div>
    <div id="panel-volatility" class="chart-panel">
      <img class="chart-img" src="data:image/png;base64,{CHART4}" alt="Rolling volatility chart"/>
      <p class="chart-caption">Figure 3b &mdash; 30-day rolling annualised volatility for Apple and Tesla (2020&ndash;2024) calculated from real daily returns.</p>
    </div>
    <div class="proj-tags"><span>Python</span><span>yfinance</span><span>Pandas</span><span>Matplotlib</span><span>Interactive</span><span>S&amp;P 500</span><span>Risk Analysis</span></div>
  </div>
</section>
<section id="education">
  <div class="section-title">Education</div>
  <div class="edu-card">
    <div class="edu-icon">&#127891;</div>
    <div>
      <h3>BSc Accounting and Finance</h3>
      <p>University &nbsp;|&nbsp; Currently Enrolled</p>
      <p style="margin-top:0.5rem;font-size:0.88rem;color:#555;">Relevant modules include Data Literacy, Financial Reporting, and Quantitative Methods. Developing applied skills in Python, data analysis, and financial modelling alongside core accounting and finance studies.</p>
    </div>
  </div>
</section>
<section id="contact">
  <div class="contact-card">
    <h2>Get In Touch</h2>
    <p>Feel free to connect with me on GitHub or reach out via email.</p>
    <div class="contact-links">
      <a href="https://github.com/{profile["github"]}" target="_blank">&#128279; GitHub Profile</a>
      <a href="https://github.com/{profile["github"]}/{profile["repo"]}" target="_blank">&#128193; Portfolio Repo</a>
    </div>
  </div>
</section>
<footer><p>&copy; 2025 {profile["name"]} &nbsp;|&nbsp; Built with Python, yfinance, Pandas &amp; Matplotlib &nbsp;|&nbsp; Deployed via GitHub Pages</p></footer>
<script>
function switchChart(val) {{
  document.querySelectorAll('.chart-panel').forEach(p => p.classList.remove('active'));
  document.getElementById('panel-' + val).classList.add('active');
}}
</script>
</body></html>'''

output = pathlib.Path('docs/index.html')
output.parent.mkdir(exist_ok=True)
output.write_text(html, encoding='utf-8')
print(f'Portfolio written: {output.resolve()}')
print(f'File size: {output.stat().st_size:,} bytes')

## Deployment

Push to GitHub and enable Pages:

```bash
git add .
git commit -m "Add data visualisation charts with real yfinance data"
git push origin main
```

Then in GitHub repo Settings > Pages > branch: main, folder: /docs

Live at: https://salhazeem.github.io/repo2-AF1204-salhazeem